In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

script = Path(
    "/content/drive/MyDrive/Allinone/scripts/"
    "run_deadlift_v6_test_20261109.py"
)

if not script.is_file():
    raise FileNotFoundError(
        "테스트 코드를 찾지 못했습니다. "
        "Allinone이 있는 Google 계정으로 Drive를 연결했는지 확인하세요."
    )

exec(
    compile(script.read_text(encoding="utf-8"), str(script), "exec"),
    {"__name__": "__main__", "__file__": str(script)}
)

In [ ]:
from google.colab import drive
from pathlib import Path
import runpy

drive.mount("/content/drive")

script = Path(
    "/content/drive/MyDrive/Allinone/scripts/"
    "Allinone_All_Exercises_20261109.py"
)

if not script.is_file():
    raise FileNotFoundError(
        "전체 운동 실행 코드를 찾지 못했습니다. "
        "Allinone 폴더가 있는 Google 계정으로 연결했는지 확인하세요."
    )

result = runpy.run_path(str(script), run_name="__main__")
PROJECT = result["PROJECT"]

In [ ]:
"""Colab: collect public reference photos for exact, currently unsupported exercises.
This is acquisition, not classifier training. Existing models/data are read-only.
Images are not videos; stages of one demo are not independent people/samples.
"""
from __future__ import annotations
import ast
import csv
import hashlib
import io
import json
import re
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from urllib.parse import quote

ROOT = Path('/content/drive/MyDrive/Allinone')
REPO = 'yuhonas/free-exercise-db'
RAW = 'https://raw.githubusercontent.com/' + REPO + '/main/'
SOURCE_PAGE = 'https://github.com/' + REPO
MAX_BYTES = 100 * 1024**2
MAX_IMAGE_BYTES = 6 * 1024**2
# Exact source IDs only. Missing/unclear mappings remain missing rather than
# substituting a dumbbell, band, incline press, or different leg-press machine.
SOURCE_IDS = {
    'treadmill_walk': ['Walking_Treadmill'],
    'treadmill_run': ['Running_Treadmill'],
    'reverse_pec_deck': ['Reverse_Machine_Flyes'],
    'overhand_lat_pulldown': ['Wide-Grip_Lat_Pulldown'],
    'neutral_lat_pulldown': ['V-Bar_Pulldown'],
    'machine_shoulder_press': ['Machine_Shoulder_Military_Press', 'Leverage_Shoulder_Press'],
    'seated_cable_row': ['Seated_Cable_Rows'],
    'machine_biceps_curl': ['Machine_Bicep_Curl', 'Machine_Preacher_Curls'],
    'single_leg_extension': ['Single-Leg_Leg_Extension'],
    'seated_leg_curl': ['Seated_Leg_Curl'],
    'cable_chest_fly': ['Cable_Crossover'],
    'cable_lateral_raise': ['Standing_Low-Pulley_Deltoid_Raise'],
    'smith_squat': ['Smith_Machine_Squat'],
    'smith_shoulder_press': ['Smith_Machine_Overhead_Shoulder_Press'],
    'smith_bench_press': ['Smith_Machine_Bench_Press'],
    'hanging_leg_raise': ['Hanging_Leg_Raise'],
    'barbell_back_squat': ['Barbell_Squat', 'Barbell_Full_Squat'],
    'barbell_lunge': ['Barbell_Lunge'],
}


def save_json(path: Path, data: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def load_catalog(path: Path) -> list[dict]:
    # Read a literal catalog without running the old environment/training/UI.
    tree = ast.parse(path.read_text(encoding='utf-8-sig'))
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'CATALOG' for t in node.targets):
            if isinstance(node.value, ast.Call) and len(node.value.args) == 1:
                value = json.loads(ast.literal_eval(node.value.args[0]))
                return [r for r in value if r.get('requested')]
    raise ValueError('기존 실행 코드의 CATALOG를 읽지 못했습니다. 모델은 변경하지 않았습니다.')


def safe_source_path(value: str) -> PurePosixPath:
    p = PurePosixPath(value)
    if ':' in value or p.is_absolute() or '..' in p.parts or '\\' in value or p.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:
        raise ValueError('허용되지 않은 원본 이미지 경로: ' + value)
    return p


def main(root: Path = ROOT) -> dict:
    if not root.is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    if not root.is_dir():
        raise FileNotFoundError('기존 Allinone 폴더가 있는 계정으로 Drive를 연결하세요.')
    import requests
    from PIL import Image

    script = root / 'scripts/Allinone_All_Exercises_20261109.py'
    catalog = load_catalog(script)
    registry_file = root / 'All_Exercises_Prototype/latest_registry.json'
    if not registry_file.is_file():
        raise FileNotFoundError('먼저 전체 운동 실행기로 기존 모델 연결을 완료하세요.')
    registry = json.loads(registry_file.read_text(encoding='utf-8'))
    connected = {r['exercise'] for r in registry.get('models', [])}
    missing = [r for r in catalog if r['id'] not in connected]
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
    base = root / 'Public_Exercise_References'
    run = base / 'runs' / run_id
    assets_dir = base / 'free_exercise_db'
    run.mkdir(parents=True, exist_ok=False)
    print(f'전체 {len(catalog)}항목 중 미지원 {len(missing)}항목의 공개 참고자료만 수집합니다.', flush=True)
    print('※ 이 단계는 사진 수집입니다. 새 모델 생성/정확도 검증/기존 모델 변경은 하지 않습니다.', flush=True)
    session = requests.Session()
    session.headers.update({'User-Agent': 'Allinone-Research-ReferenceCollector/1.0'})
    failures: list[dict] = []
    rows: list[dict] = []
    manifest: list[dict] = []
    transferred = 0
    try:
        print('[1/2] 공개 운동 목록·라이선스 확인', flush=True)
        response = session.get(RAW + 'dist/exercises.json', timeout=(15, 90))
        response.raise_for_status()
        if len(response.content) > 12 * 1024**2:
            raise ValueError('목록 크기가 예상 범위를 초과했습니다.')
        data = response.json()
        if not isinstance(data, list) or not all(isinstance(r, dict) for r in data):
            raise ValueError('공개 목록의 형식이 변경되었습니다.')
        save_json(run / 'source_catalog.json', data)
        license_response = session.get(RAW + 'LICENSE.md', timeout=(15, 45))
        license_response.raise_for_status()
        license_text = license_response.text
        if not ('public domain' in license_text.lower() and 'unlicense' in license_text.lower()):
            raise RuntimeError('배포 라이선스가 확인 당시와 다릅니다. 사진 수집을 중단합니다.')
        (run / 'SOURCE_LICENSE.md').write_text(license_text, encoding='utf-8')
        save_json(run / 'source_info.json', {
            'repository': SOURCE_PAGE, 'catalog_url': RAW + 'dist/exercises.json',
            'catalog_sha256': hashlib.sha256(response.content).hexdigest(),
            'fetched_at_utc': datetime.now(timezone.utc).isoformat(),
            'license_note': 'Repository declaration: Unlicense/public domain; source notice retained.',
            'no_explicit_quality_labels': True,
            'purpose': 'Reference acquisition; not independent real-world GOOD/BAD validation.',
        })
        index = {str(r.get('id')): r for r in data}
        print('[2/2] 미지원 운동별 사진 수집', flush=True)
        for i, target in enumerate(missing, 1):
            ex = target['id']; hashes: set[str] = set(); groups: set[str] = set()
            source_ids = SOURCE_IDS.get(ex, [])
            for source_id in source_ids:
                record = index.get(source_id)
                if record is None:
                    failures.append({'exercise': ex, 'source_id': source_id, 'reason': 'SOURCE_ID_NOT_IN_CATALOG'})
                    continue
                if not re.fullmatch(r'[A-Za-z0-9_-]+', source_id):
                    failures.append({'exercise': ex, 'source_id': source_id, 'reason': 'UNSAFE_SOURCE_ID'})
                    continue
                folder = assets_dir / ex / source_id
                folder.mkdir(parents=True, exist_ok=True)
                save_json(folder / 'source_record.json', {
                    'source': SOURCE_PAGE, 'exercise_id': ex, 'original_record': record,
                    'use': 'reference_only', 'quality': 'GOOD',
                    'label_origin': 'assumed_good_unlabeled',
                    'subject_id': None,
                    'group': 'free-exercise-db:' + source_id,
                    'warning': '동일 동작 시범의 여러 사진이며 서로 다른 사람/독립 영상으로 세지 않음.',
                })
                for image_path in record.get('images', []):
                    url = ''
                    try:
                        relative = safe_source_path(str(image_path))
                        if not relative.parts or relative.parts[0] != source_id:
                            raise ValueError('이미지와 원본 운동 ID가 일치하지 않습니다.')
                        url = RAW + 'exercises/' + quote(relative.as_posix(), safe='/')
                        file_path = folder / relative.name
                        reused = file_path.is_file()
                        if reused and file_path.stat().st_size > MAX_IMAGE_BYTES:
                            raise ValueError('기존 참고 이미지가 파일 크기 상한을 초과합니다.')
                        content = file_path.read_bytes() if reused else None
                        if content is None:
                            if transferred >= MAX_BYTES:
                                raise ValueError('이 실행의 다운로드 상한에 도달했습니다.')
                            with session.get(url, stream=True, timeout=(15, 60)) as download:
                                download.raise_for_status()
                                chunks = []; size = 0
                                for chunk in download.iter_content(64 * 1024):
                                    if not chunk: continue
                                    size += len(chunk); transferred += len(chunk)
                                    if size > MAX_IMAGE_BYTES or transferred > MAX_BYTES:
                                        raise ValueError('다운로드 용량 상한에 도달했습니다.')
                                    chunks.append(chunk)
                                content = b''.join(chunks)
                        with Image.open(io.BytesIO(content)) as im:
                            im.verify()
                        h = hashlib.sha256(content).hexdigest()
                        if not reused:
                            tmp = file_path.with_name(file_path.name + '.part')
                            tmp.write_bytes(content); tmp.replace(file_path)
                        if h in hashes:
                            failures.append({'exercise': ex, 'url': url, 'reason': 'EXACT_DUPLICATE_NOT_COUNTED'})
                            continue
                        hashes.add(h); groups.add(source_id)
                        manifest.append({
                            'exercise_id': ex, 'equipment': target['equipment'], 'exercise_ko': target['name'],
                            'path': str(file_path), 'media_type': 'IMAGE', 'quality': 'GOOD',
                            'label_origin': 'assumed_good_unlabeled', 'source_item_id': source_id,
                            'group': 'free-exercise-db:' + source_id, 'subject_id': None,
                            'sha256': h, 'source_url': url, 'reference_only': True,
                            'pose_extracted': False, 'model_trained': False, 'reused': reused,
                        })
                        if not reused: time.sleep(0.08)
                    except Exception as exc:
                        failures.append({'exercise': ex, 'source_id': source_id, 'url': url, 'reason': str(exc)})
            status = 'REFERENCE_PHOTOS_COLLECTED_NOT_TRAINED' if hashes else 'NO_EXACT_PUBLIC_REFERENCE_COLLECTED'
            rows.append({'exercise_id': ex, '기구': target['equipment'], '운동': target['name'],
                         '사진수': len(hashes), '원본시범항목수_사람수아님': len(groups),
                         '상태': status, '추가모델수': 0})
            print(f'[{i}/{len(missing)}] {target["name"]}: 사진 {len(hashes)}장 / 원본 항목 {len(groups)} / 모델 추가 없음', flush=True)
            save_json(run / 'asset_manifest.json', manifest)
            save_json(run / 'failures.json', failures)
        with (run / 'acquisition_report.csv').open('w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['exercise_id','기구','운동','사진수','원본시범항목수_사람수아님','상태','추가모델수'])
            writer.writeheader(); writer.writerows(rows)
        summary = {
            'status': 'REFERENCE_ACQUISITION_FINISHED',
            'target_missing_items': len(missing),
            'items_with_reference_photos': sum(r['사진수'] > 0 for r in rows),
            'unique_photo_files_per_target_total': len(manifest),
            'new_models_trained': 0,
            'existing_registry_unchanged': True,
            'media_downloaded_MiB_this_run': round(transferred / 1024**2, 3),
            'failure_records': len(failures), 'output_dir': str(run),
            'note': '시범 사진 몇 장을 독립 학습 표본으로 부풀리지 않습니다. 자료가 없는 세부 동작은 계속 미지원입니다.',
        }
        save_json(run / 'summary.json', summary)
        print('\n===== 미지원 운동 공개 참고자료 수집 결과 =====', flush=True)
        print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)
        print('기존 실행기 재실행은 아직 필요 없습니다. 모델 연결 수는 이 단계에서 늘지 않습니다.', flush=True)
        return summary
    except Exception as exc:
        save_json(run / 'failure.json', {'status': 'ACQUISITION_FAILED', 'error': str(exc), 'model_changes': False})
        print('수집 중단:', str(exc), '\n기존 모델은 변경하지 않았습니다. 로그:', run, flush=True)
        raise
    finally:
        session.close()


if __name__ == '__main__':
    RESULT = main()
